In [ ]:
%%shell
apt-get update && apt-get install -y imagemagick
# Run this line to check if imagemagick is installed and find its path
# !which convert


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.special import jv, jn_zeros
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display, Markdown, HTML # Para ver la animación en Jupyter
from scipy.integrate import trapezoid # Import the recommended function
from matplotlib.colors import PowerNorm # Import PowerNorm for non-linear color scaling

# Aumentar el límite de tamaño de la animación a 50 MB
plt.rcParams['animation.embed_limit'] = 50

class DiffusionVisualizer:
    def __init__(self, D=0.5):
        self.D = D

    def animate_ring(self, L=5, frames=400, n_terms=20):
        # This is the original animate_ring that plots c vs x (arc length)
        fig, ax = plt.subplots(figsize=(9, 5))
        x = np.linspace(0, 2*L, 500)
        line, = ax.plot(x, np.zeros_like(x), lw=2, color='#2ecc71')
        ax.set_ylim(0, 1.1) # Adjusted y-limits to start from 0 after clamping
        ax.set_title("Ring Diffusion: Concentration vs. Arc Length", fontsize=12, y=1.05)
        ax.set_xlabel("Arc Length (x)")
        ax.set_ylabel("Concentration (c)")
        ax.grid(True)

        t_final = 15.0 # Desired final time for the animation
        t_step = (t_final - 0.01) / frames # Calculate step to reach t_final

        def update(frame):
            t = frame * t_step + 0.01 # Evitar t=0
            c = np.zeros_like(x)
            for n in range(n_terms):
                kn = (n + 0.5) * np.pi / (2 * L)
                decay = np.exp(-(kn**2) * self.D * t)
                term = (np.sin(kn * x) + ((-1)**n) * np.cos(kn * x)) * decay
                c += term

            # Apply user's request: exclude negative concentrations and normalize
            c = np.maximum(0, c) # Clamps all negative values to 0

            max_c_after_clamping = np.max(c) # Max after clamping
            if max_c_after_clamping < 1e-9: # If all values are essentially zero
                line.set_ydata(np.zeros_like(x))
            else:
                line.set_ydata(c / max_c_after_clamping) # Normalize by the new max

            ax.set_title(f"Ring Diffusion: Concentration vs. Arc Length (t={t:.2f})", fontsize=12, y=1.05)
            return line,

        ani = FuncAnimation(fig, update, frames=frames, interval=50, blit=True)
        plt.close(fig)
        return ani

    def animate_ring_polar(self, L=5, frames=200, n_terms=20):
        # This new animation plots concentration on a ring in polar coordinates
        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
        x = np.linspace(0, 2*L, 200) # Grid along the arc length
        angles = (x / (2 * L)) * (2 * np.pi)

        inner_radius = 0.8
        outer_radius = 1.0
        radii_mesh_points = np.linspace(inner_radius, outer_radius, 2)
        THETA_mesh, R_mesh = np.meshgrid(angles, radii_mesh_points)

        # Initial concentration for the first frame
        initial_c = np.zeros_like(x)
        t_initial = 0.01
        for n in range(n_terms):
            kn = (n + 0.5) * np.pi / (2 * L)
            decay = np.exp(-(kn**2) * self.D * t_initial)
            angular_part = np.sin(kn * x) + ((-1)**n) * np.cos(kn * x)
            initial_c += angular_part * decay

        # Normalize initial concentration for color mapping
        if np.max(np.abs(initial_c)) > 1e-9:
            initial_c = initial_c / np.max(np.abs(initial_c))

        C_mesh_initial = np.tile(initial_c, (2, 1))

        # Fixed color scale for normalized concentration [0, 1] (adjusted from [-1, 1])
        mesh = ax.pcolormesh(THETA_mesh, R_mesh, C_mesh_initial, shading='gouraud', cmap='viridis', vmin=0, vmax=1)
        fig.colorbar(mesh, ax=ax, orientation='vertical', label='Concentration c') # Added color bar

        ax.set_theta_zero_location('N') # Set 0 degrees to the top
        ax.set_theta_direction(-1) # Go clockwise
        ax.set_rticks(np.linspace(inner_radius, outer_radius, 3)) # Add radial ticks
        ax.set_rlabel_position(45) # Position radial labels
        ax.set_thetagrids(np.arange(0, 360, 45)) # Add angular grid lines

        t_final = 15.0 # Desired final time for the animation
        t_step = (t_final - 0.01) / frames # Calculate step to reach t_final

        def update(frame):
            t = frame * t_step + 0.01
            c = np.zeros_like(x)
            for n in range(n_terms):
                kn = (n + 0.5) * np.pi / (2 * L)
                decay = np.exp(-(kn**2) * self.D * t)
                angular_part = np.sin(kn * x) + ((-1)**n) * np.cos(kn * x)
                c += angular_part * decay

            # Normalize for consistent color scaling across frames
            if np.max(np.abs(c)) > 1e-9:
                c_normalized = c / np.max(np.abs(c))
            else:
                c_normalized = np.zeros_like(c)

            C_mesh = np.tile(c_normalized, (2, 1))
            mesh.set_array(C_mesh.ravel())
            # Color limits are now fixed, no set_clim here

            ax.set_title(f"Ring Diffusion (Polar Animation) L={L:.1f} | t={t:.2f}", fontsize=12, y=1.05)
            return mesh,

        ani = FuncAnimation(fig, update, frames=frames, interval=100, blit=False)
        plt.close(fig)
        return ani

    def animate_wedge(self, theta_0_rad=np.pi/3, r_max=5, frames=200, n_modes=3, k_zeros=5):
        fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(6, 6))
        r = np.linspace(0.1, r_max, 50)
        theta = np.linspace(0, theta_0_rad, 50)
        R, THETA = np.meshgrid(r, theta)

        initial_P = np.zeros_like(R)
        t_initial = 0.01
        for n_val in range(1, n_modes + 1):
            nu = (n_val * np.pi) / theta_0_rad
            # Ensure nu is an integer type, as jn_zeros expects it
            # This is mathematically valid if theta_0_rad is chosen such that (n_val * pi) / theta_0_rad is an integer
            try:
                zeros = jn_zeros(int(nu), k_zeros)
            except ValueError: # Catch error if jn_zeros fails for non-integer nu or invalid input
                zeros = [] # Treat as no zeros found for this mode
            for j_nk in zeros:
                alpha = j_nk / r_max
                decay = np.exp(-(alpha**2) * self.D * t_initial)
                initial_P += np.sin(nu * THETA) * jv(nu, alpha * R) * decay

        initial_P[np.isnan(initial_P)] = 0.0
        initial_P[np.isinf(initial_P)] = 0.0
        initial_P = np.maximum(0, initial_P) # Clamp negative concentrations to 0

        # Define vmax based on the initial concentration to keep the scale fixed
        fixed_vmax_wedge = np.max(initial_P)
        if fixed_vmax_wedge < 1e-9: # Handle case where initial P is essentially zero
            fixed_vmax_wedge = 1.0 # Default max if all values are tiny

        # Use PowerNorm to emphasize differences at lower concentrations (gamma < 1)
        # gamma = 0.5 is like square root scaling. vmin is fixed at 0.
        norm = PowerNorm(gamma=0.5, vmin=0, vmax=fixed_vmax_wedge)
        cont = ax.pcolormesh(THETA, R, initial_P, shading='gouraud', cmap='viridis', norm=norm)
        fig.colorbar(cont, ax=ax, orientation='vertical', label='Concentration P') # Added color bar

        # Add radial ticks and position, consistent with interactive wedge plot
        ax.set_rticks(np.linspace(r[0], r_max, 4))
        ax.set_rlabel_position(-22.5)
        ax.set_thetagrids([]) # Thetamax/thetamin define the angular extent

        t_final = 15.0 # Desired final time for the animation
        t_step = (t_final - 0.01) / frames # Calculate step to reach t_final

        def update(frame):
            t = frame * t_step + 0.01
            P = np.zeros_like(R)
            for n_val in range(1, n_modes + 1):
                nu = (n_val * np.pi) / theta_0_rad
                try:
                    zeros = jn_zeros(int(nu), k_zeros)
                except ValueError:
                    zeros = []
                for j_nk in zeros:
                    alpha = j_nk / r_max
                    decay = np.exp(-(alpha**2) * self.D * t)
                    P += np.sin(nu * THETA) * jv(nu, alpha * R) * decay

            P[np.isnan(P)] = 0.0
            P[np.isinf(P)] = 0.0
            P = np.maximum(0, P) # Clamp negative concentrations to 0

            cont.set_array(P.ravel()) # Update the array
            # Color limits are now fixed by PowerNorm, no dynamic set_clim or norm.vmax here

            ax.set_thetamin(0)
            ax.set_thetamax(np.degrees(theta_0_rad))
            # Use similar title format as code 1 for uniformity
            ax.set_title(f"Wedge Diffusion (Absorbing Boundaries)\nθ={np.degrees(theta_0_rad):.1f}° | $r_{{max}}$={r_max:.1f} | t={t:.1f}", fontsize=12, y=1.05)
            return cont,

        ani = FuncAnimation(fig, update, frames=frames, interval=100, blit=False)
        plt.close(fig)
        return ani

    def animate_wedge_survival_function(self, theta_0_rad=np.pi/3, r_max=5, frames=200, n_modes=10, k_zeros=15):
        fig, ax = plt.subplots(figsize=(8, 5))
        line, = ax.plot([], [], lw=2, color='blue')
        ax.set_xlim(0, 15.0) # Max time for the animation
        ax.set_xlabel("Time (t)")
        ax.set_ylabel("Survival Function S(t) (Normalized)") # Updated label
        ax.set_title(f"Wedge Diffusion: Survival Function S(t)\nθ={np.degrees(theta_0_rad):.1f}° | $r_{{max}}$={r_max:.1f}", fontsize=12, y=1.05) # Updated title
        ax.grid(True)

        t_final = 15.0 # Desired final time for the animation
        t_step = (t_final - 0.01) / frames

        times = []
        survival_values = []

        r_grid_P = np.linspace(0.1, r_max, 50) # Decreased resolution for speed
        theta_grid_P = np.linspace(0, theta_0_rad, 50) # Decreased resolution for speed
        R_mesh_P, THETA_mesh_P = np.meshgrid(r_grid_P, theta_grid_P)

        # Calculate the initial total quantity to set y-axis limit for normalization
        r_int = np.linspace(0.1, r_max, 100)
        theta_int = np.linspace(0, theta_0_rad, 100)
        R_int_mesh, THETA_int_mesh = np.meshgrid(r_int, theta_int)

        P_at_t0 = np.zeros_like(R_int_mesh)
        t_for_initial_calc = 0.01 # Calculate initial quantity at a small time close to 0

        for n_val in range(1, n_modes + 1):
            nu = (n_val * np.pi) / theta_0_rad
            try:
                zeros = jn_zeros(int(nu), k_zeros)
            except ValueError:
                zeros = []
            for j_nk in zeros:
                alpha = j_nk / r_max
                decay_factor = np.exp(-(alpha**2) * self.D * t_for_initial_calc)
                term = np.sin(nu * THETA_int_mesh) * jv(nu, alpha * R_int_mesh) * decay_factor
                P_at_t0 += term

        P_at_t0[np.isnan(P_at_t0)] = 0.0
        P_at_t0[np.isinf(P_at_t0)] = 0.0
        P_at_t0 = np.maximum(0, P_at_t0) # Clamp negative concentrations to 0

        # Numerical integration of P * r d(theta) dr
        integral_r_val = trapezoid(P_at_t0 * R_int_mesh, x=r_int, axis=1) # Integrate over r
        initial_absolute_quantity = trapezoid(integral_r_val, x=theta_int) # Integrate over theta

        # Set y-axis limits for normalized S(t) (0 to 1)
        if initial_absolute_quantity < 1e-9: # If initial quantity is essentially zero
            ax.set_ylim(0, 1.1) # Default small range, slightly above 1
            print(f"Warning: Initial absolute quantity is very close to zero ({initial_absolute_quantity:.2e}). Y-axis will be set to (0, 1.1).")
        else:
            ax.set_ylim(0, 1.1) # S(t) will range from ~1 to 0
            print(f"Initial absolute quantity: {initial_absolute_quantity:.4f}. Y-axis will be set to (0, 1.1) for normalized S(t).")

        def update(frame):
            t = frame * t_step + 0.01
            current_P = np.zeros_like(R_mesh_P)
            for n_val in range(1, n_modes + 1):
                nu = (n_val * np.pi) / theta_0_rad
                try:
                    zeros = jn_zeros(int(nu), k_zeros)
                except ValueError:
                    zeros = []
                for j_nk in zeros:
                    alpha = j_nk / r_max
                    decay = np.exp(-(alpha**2) * self.D * t)
                    current_P += np.sin(nu * THETA_mesh_P) * jv(nu, alpha * R_mesh_P) * decay

            current_P[np.isnan(current_P)] = 0.0
            current_P[np.isinf(current_P)] = 0.0
            current_P = np.maximum(0, current_P) # Clamp negative concentrations to 0

            integral_r = trapezoid(current_P * R_mesh_P, x=r_grid_P, axis=1)
            current_amount = trapezoid(integral_r, x=theta_grid_P)

            # Normalize the current amount by the initial amount to get S(t)
            if initial_absolute_quantity > 1e-9:
                s_t_value = current_amount / initial_absolute_quantity
            else:
                s_t_value = 0.0 # If initial is zero, S(t) is zero

            times.append(t)
            survival_values.append(s_t_value)

            line.set_data(times, survival_values)
            ax.set_title(f"Wedge Diffusion: Survival Function S(t)\nθ={np.degrees(theta_0_rad):.1f}° | $r_{{max}}$={r_max:.1f} | t={t:.2f}", fontsize=12, y=1.05)
            # No longer printing current amount, but the S(t) value would be here if needed.
            return line,

        ani = FuncAnimation(fig, update, frames=frames, interval=100, blit=False)
        plt.close(fig)
        return ani

diffusion_sim.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv, jn_zeros
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display, Markdown
from matplotlib.colors import PowerNorm # Import PowerNorm

def simulate_wedge(theta_0_rad, r_max, t, n_modes=5, k_zeros=10):
    """
    Solves Diffusion in a Wedge with absorbing boundaries.
    Based on Separation of Variables: P(r,theta,t) = R(r)Theta(theta)T(t)
    """
    # Grid in polar coordinates
    r = np.linspace(0.1, r_max, 400) # Increased resolution for better stability at small angles
    theta = np.linspace(0, theta_0_rad, 400) # Increased resolution for better stability at small angles
    R, THETA = np.meshgrid(r, theta)

    P = np.zeros_like(R)
    D = 1.0 # Diffusion coefficient

    for n in range(1, n_modes + 1):
        nu = (n * np.pi) / theta_0_rad # Angular eigenvalue from PDF pg. 4
        # Get the first 'k' zeros of the Bessel function of order nu
        # jn_zeros expects an integer order. If nu is not integer, int(nu) truncates.
        # This can lead to numerical instability or errors.
        try:
            zeros = jn_zeros(int(nu), k_zeros)
        except ValueError: # Catch error if jn_zeros fails for non-integer nu or invalid input
            zeros = [] # Treat as no zeros found for this mode

        for j_nk in zeros:
            # Eigenvalues for time decay: lambda = (j_nk / r_max)^2
            alpha = j_nk / r_max
            decay = np.exp(-(alpha**2) * D * t)
            # Full solution: sin(angular) * J_nu(radial) * exp(time)
            term = np.sin(nu * THETA) * jv(nu, alpha * R) * decay
            P += term

    P[np.isnan(P)] = 0.0 # Replace any NaN values with 0
    P[np.isinf(P)] = 0.0 # Also replace any Inf values with 0
    P = np.maximum(0, P) # Clamp negative concentrations to 0 as concentration should be non-negative
    return R, THETA, P

def plot_wedge_diffusion(theta_0_deg, r_max, t, n_modes=5, k_zeros=10):
    """
    Plots the diffusion in a wedge with absorbing boundaries.

    Parameters:
    theta_0_deg (float): The initial angle of the wedge in degrees.
    r_max (float): The maximum radial extent of the wedge.
    t (float): The time at which to calculate the diffusion.
    n_modes (int): Number of angular modes to include in the simulation. (Default: 3)
    k_zeros (int): Number of Bessel function zeros to include for each mode. (Default: 5)
    """
    # Convert theta_0 from degrees to radians for simulation
    theta_0_rad = np.deg2rad(theta_0_deg)

    # Visualization logic (Polar Plot)
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

    # Try to simulate and plot, catching potential errors
    try:
        R_val, THETA_val, P_val = simulate_wedge(theta_0_rad=theta_0_rad, r_max=r_max, t=t, n_modes=n_modes, k_zeros=k_zeros)

        # Check for any remaining NaN or Inf values after simulation
        if np.any(np.isnan(P_val)) or np.any(np.isinf(P_val)):
            plt.close(fig) # Close the figure to prevent empty plot
            display(Markdown(f"**Error/Warning**: Numerical instability detected for parameters: θ={theta_0_deg:.1f}° | $r_{{max}}$={r_max:.1f} | t={t:.1f}. The concentration array contains `NaN` or `Inf` values even after internal handling. Please try adjusting the parameters, especially θ values."))
            return

        # Set vmin and vmax for the colormap. Clamp to 0 as min.
        vmin_plot = 0.0
        vmax_plot = np.max(P_val)
        if vmax_plot < 1e-9: # Prevent division by zero or too small range
            vmax_plot = 1.0

        # Use linear scaling instead of PowerNorm
        mesh = ax.pcolormesh(THETA_val, R_val, P_val, shading='gouraud', cmap='viridis', vmin=vmin_plot, vmax=vmax_plot)
        fig.colorbar(mesh, ax=ax, orientation='vertical', label='Concentration P')

        # Add radial grid lines for better readability
        ax.set_rticks(np.linspace(r_max/4, r_max, 4)) # Show ticks at 1/4, 1/2, 3/4, full r_max
        ax.set_rlabel_position(-22.5) # Position radial labels slightly offset

        # Refined title for clarity and to reduce potential overlapping
        ax.set_title(f"Wedge Diffusion (Absorbing Boundaries)\nθ={theta_0_deg:.1f}° | $r_{{max}}$={r_max:.1f} | t={t:.1f}", fontsize=12, y=1.05)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        plt.close(fig) # Close the figure to prevent empty plot
        display(Markdown(f"**Error**: An unexpected error occurred during simulation or plotting with parameters: θ={theta_0_deg:.1f}° | $r_{{max}}$={r_max:.1f} | t={t:.1f}. Error message: `{e}`. Please try adjusting the parameters or consider the numerical limitations for certain angles."))
        return

    if theta_0_deg < 20:
        display(Markdown("**Warning**: For very small angles (θ < 20°), the simulation might show visual artifacts due to rapid oscillations of the solution. Increasing grid resolution for extremely small angles can be computationally intensive."))

# Create interactive sliders for the parameters
interact(plot_wedge_diffusion,
         theta_0_deg=FloatSlider(min=15, max=360, step=1, value=45, description='θ (degrees)'),
         r_max=FloatSlider(min=1, max=20, step=0.5, value=5, description='r_max'),
         t=FloatSlider(min=0.01, max=15, step=0.1, value=0.5, description='Time (t)'));

interactive(children=(FloatSlider(value=45.0, description='θ (degrees)', max=360.0, min=15.0, step=1.0), Float…

In [ ]:
viz = DiffusionVisualizer(D=0.2)
# Para ver la Cuña:
# Increased r_max for a better video as requested, and increased frames
wedge_ani = viz.animate_wedge(theta_0_rad=np.pi/3, r_max=15, frames=100, n_modes=10, k_zeros=15)
HTML(wedge_ani.to_jshtml())

# Guardar la animación como GIF y MP4
# Asegúrate de tener 'imagemagick' y 'ffmpeg' instalados en tu sistema para esto.
# En Colab, usualmente ya están disponibles.
wedge_ani.save('wedge_diffusion.gif', writer='imagemagick', fps=10)
wedge_ani.save('wedge_diffusion.mp4', writer='ffmpeg', fps=10)

In [ ]:
import matplotlib as mpl
# Set the path to the imagemagick 'convert' executable if it's not automatically found
# You might need to adjust this path based on the output of '!which convert' from the previous cell
mpl.rcParams['animation.convert_path'] = '/usr/bin/convert'

print("Imagemagick configurado para matplotlib. Por favor, vuelve a ejecutar las celdas de animación para aplicar el cambio.")

Imagemagick configurado para matplotlib. Por favor, vuelve a ejecutar las celdas de animación para aplicar el cambio.


In [ ]:
viz = DiffusionVisualizer(D=0.2) # Set D back to 0.2
# Para ver la Función de Supervivencia de la Cuña:
# Using default n_modes=10 and k_zeros=15 as requested
survival_ani = viz.animate_wedge_survival_function(theta_0_rad=np.pi/3, r_max=5, frames=200, n_modes=10, k_zeros=15)
HTML(survival_ani.to_jshtml())

# Guardar la animación como GIF y MP4
survival_ani.save('wedge_survival.gif', writer='imagemagick', fps=10)
survival_ani.save('wedge_survival.mp4', writer='ffmpeg', fps=10)

Initial absolute quantity: 5.8244. Y-axis will be set to (0, 1.1) for normalized S(t).


ring_diffusion.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display, Markdown

def simulate_ring(L, t, D=1.0, n_terms=20):
    """
    Solves Diffusion in a circular ring (circumference 2L).
    Periodic BCs: c(0,t) = c(2L,t) and cx(0,t) = cx(2L,t)

    Parameters:
    L (float): Half of the circumference of the ring (total circumference is 2L).
    t (float): Time at which to calculate the diffusion.
    D (float): Diffusion coefficient. Default is 1.0.
    n_terms (int): Number of terms to include in the Fourier series sum. Default is 20.
    """
    x = np.linspace(0, 2*L, 500) # Grid along the arc length
    concentration = np.zeros_like(x)

    # Using the solution from PDF pg. 11
    # c(x,t) = sum Gn * [sin(kn*x) + (-1)^n * cos(kn*x)] * exp(-lambda*D*t)
    # Note: The initial condition G_n coefficients are assumed to be 1 for simplicity
    # for demonstration, representing a 'general' solution. A specific initial condition
    # would determine the actual G_n coefficients.

    for n in range(n_terms):
        kn = (n + 0.5) * np.pi / (2 * L)
        decay = np.exp(-(kn**2) * D * t)

        # Eigenfunction Xn(x)
        angular_part = np.sin(kn * x) + ((-1)**n) * np.cos(kn * x)
        concentration += angular_part * decay

    return x, concentration

def plot_ring_diffusion(L, t, D, n_terms):
    """
    Plots the diffusion in a ring with periodic boundary conditions.

    Parameters:
    L (float): Half of the circumference of the ring.
    t (float): Time at which to calculate the diffusion.
    D (float): Diffusion coefficient.
    n_terms (int): Number of terms to include in the Fourier series sum.
    """
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

    try:
        x_val, c_val = simulate_ring(L=L, t=t, D=D, n_terms=n_terms)

        # Map x_val (arc length) to an angle for polar plot
        # x ranges from 0 to 2L, so angle ranges from 0 to 2*pi
        angles = (x_val / (2 * L)) * (2 * np.pi)

        # Normalize concentration for color mapping if desired, or use directly
        # For plotting as a 'heatmap' on a ring, we can use pcolormesh.
        # To visualize it as a filled ring, we can create an inner and outer radius.
        inner_radius = 0.8
        outer_radius = 1.0
        radii = np.linspace(inner_radius, outer_radius, 2) # Create a small radial extent
        # Create a mesh for pcolormesh in polar coordinates
        THETA_mesh, R_mesh = np.meshgrid(angles, radii)
        C_mesh = np.tile(c_val, (2, 1)) # Tile concentration for the radial extent

        mesh = ax.pcolormesh(THETA_mesh, R_mesh, C_mesh, shading='gouraud', cmap='viridis', vmin=np.min(c_val), vmax=np.max(c_val))
        fig.colorbar(mesh, ax=ax, orientation='vertical', label='Concentration c')

        ax.set_theta_zero_location('N') # Set 0 degrees to the top
        ax.set_theta_direction(-1) # Go clockwise
        # Re-enable radial ticks to show coordinate axes
        ax.set_rticks(np.linspace(inner_radius, outer_radius, 3)) # Show ticks at inner, middle, and outer radius
        ax.set_rlabel_position(45) # Position radial labels
        # Re-enable angular grid lines
        ax.set_thetagrids(np.arange(0, 360, 45)) # Show angular grid lines every 45 degrees

        ax.set_title(f"Ring Diffusion (Periodic BCs)\nL={L:.1f} | t={t:.1f} | D={D:.1f}", fontsize=12, y=1.05)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        plt.close(fig) # Close the figure to prevent empty plot
        display(Markdown(f"**Error**: An unexpected error occurred during simulation or plotting with parameters: L={L:.1f} | t={t:.1f} | D={D:.1f}. Error message: `{e}`. Please try adjusting the parameters."))
        return

# Create interactive sliders for the parameters
interact(plot_ring_diffusion,
         L=FloatSlider(min=1, max=10, step=0.5, value=5, description='L (Half Circumference)'),
         t=FloatSlider(min=0.01, max=5, step=0.1, value=0.1, description='Time (t)'),
         D=FloatSlider(min=0.1, max=5, step=0.1, value=1.0, description='Diffusion Coeff (D)'),
         n_terms=IntSlider(min=5, max=100, step=5, value=20, description='Number of Terms'));

In [ ]:
viz = DiffusionVisualizer(D=0.1)
# Para ver el Anillo:
# Increased n_terms to potentially get a more visible initial sum, and frames for longer animation
ring_cartesian_ani = viz.animate_ring(L=25, frames=100, n_terms=20)
HTML(ring_cartesian_ani.to_jshtml())

# Guardar la animación como GIF y MP4
ring_cartesian_ani.save('ring_diffusion_cartesian.gif', writer='imagemagick', fps=10)
ring_cartesian_ani.save('ring_diffusion_cartesian.mp4', writer='ffmpeg', fps=10)

In [ ]:
viz = DiffusionVisualizer(D=0.1)
# Para ver la nueva animación del Anillo en coordenadas polares:
ring_polar_ani = viz.animate_ring_polar(L=5, frames=100, n_terms=20)
HTML(ring_polar_ani.to_jshtml())

# Guardar la animación como GIF y MP4
ring_polar_ani.save('ring_diffusion_polar.gif', writer='imagemagick', fps=10)
ring_polar_ani.save('ring_diffusion_polar.mp4', writer='ffmpeg', fps=10)